In [ ]:
import os
from dotenv import load_dotenv; load_dotenv()
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
from sqlalchemy import create_engine



In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Sample DataFrame with multiple categorical columns
df = pd.DataFrame({
    'Age': [25, 30, 45, 22],
    'City': ['New York', 'Paris', 'Tokyo', 'Paris'],
    'Device': ['Mobile', 'Desktop', 'Mobile', 'Tablet']
})
print(df)

# 1. Define your list of columns
categorical_cols = ['City', 'Device']

# 2. Fit and transform just those columns
encoder = OneHotEncoder(sparse_output=False, dtype=int)
encoded_array = encoder.fit_transform(df[categorical_cols])

# 3. Create a DataFrame from the encoded array with the correct feature names
df_encoded_cols = pd.DataFrame(
    encoded_array, 
    columns=encoder.get_feature_names_out(categorical_cols)
)

# 4. Drop the original categorical columns and concat the new encoded ones
df_final = pd.concat([df.drop(columns=categorical_cols), df_encoded_cols], axis=1)

print(df_final)

In [5]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

df = pd.DataFrame({
    'Age': [25, 30, 45, 22],
    'City': ['New York', 'Paris', 'Tokyo', 'Paris'],
    'Device': ['Mobile', 'Desktop', 'Mobile', 'Tablet']
})

# 1. Define the transformer specifying the list of columns
# 'remainder='passthrough'' ensures numerical columns like 'Age' aren't dropped
ct = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(sparse_output=False, dtype=int), ['City', 'Device'])
    ], 
    remainder='passthrough'
)

# 2. Fit and transform the DataFrame
# Note: ColumnTransformer returns a numpy array, and changes column order (encoded columns first)
encoded_array = ct.fit_transform(df)

# 3. Rebuild the DataFrame using the transformer's tracking of column names
df_final = pd.DataFrame(encoded_array, columns=ct.get_feature_names_out())

# Clean up column names to look nicer (removes the 'onehot__' and 'remainder__' prefixes)
df_final.columns = df_final.columns.str.replace('onehot__', '').str.replace('remainder__', '')

df_final

,City_New York,City_Paris,City_Tokyo,Device_Desktop,Device_Mobile,Device_Tablet,Age
0,1,0,0,0,1,0,25
1,0,1,0,1,0,0,30
2,0,0,1,0,1,0,45
3,0,1,0,0,0,1,22


In [6]:
df

,Age,City,Device
0,25,New York,Mobile
1,30,Paris,Desktop
2,45,Tokyo,Mobile
3,22,Paris,Tablet


In [ ]:
class SQLDataset(Dataset):
    def __init__(self, connection_string, query, target_column):
        engin = create_engine(connection_string)
        self.data = pd.read_sql(query, engin)

        self.y = self.data[target_column].values
        self.X = self.data.drop(columns=target_column).values

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        features = torch.tensor(self.X[index], dtype=torch.float32)
        label = torch.tensor(self.y[index], dtype=torch.float32)
        return features, label


In [ ]:
from torch.utils.data import IterableDataset

class SQLStreamDataset(IterableDataset):
    def __init__(self, connection_string, query, batch_size=1000):
        self.engine = create_engine(connection_string)
        self.query = query
        self.batch_size = batch_size

    def __iter__(self):
        # Streams data from the database without loading the whole table into RAM
        with self.engine.connect() as conn:
            # execution_options stream results row by row or batch by batch
            result_stream = conn.execution_options(stream_results=True).execute(self.query)
            
            while True:
                chunk = result_stream.fetchmany(self.batch_size)
                if not chunk:
                    break
                for row in chunk:
                    # Parse features and label from the row
                    features = torch.tensor(row[:-1], dtype=torch.float32)
                    label = torch.tensor(row[-1], dtype=torch.float32)
                    yield features, label

In [ ]:
# Database connection details
DATABASE_URL=os.getenv("DATABASE_URL")
QUERY = 'SELECT feature1, feature2, feature3, target_label FROM your_table_name;'

# Initialize the dataset
sql_dataset = SQLDataset(connection_string=DATABASE_URL, query=QUERY, target_column='target_label')

total_size = len(sql_dataset)
train_size = int(0.70 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

print(f"Total: {total_size} | Train: {train_size} | Val: {val_size} | Test: {test_size}")

# Set seed for reproducible splits
generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset, test_dataset = random_split(
    sql_dataset, 
    [train_size, val_size, test_size],
    generator=generator
)

# Create the DataLoader (e.g., batches of 32 rows, shuffled)
train_loader = DataLoader(sql_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
epochs = 10

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train() # Set model to training mode
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    # --- VALIDATION PHASE (EVAL) ---
    model.eval() # Set model to evaluation mode (disables dropout/batchnorm)
    val_loss = 0.0
    
    with torch.no_grad(): # Disables gradient calculation to save memory and speed up
        for batch_X, batch_y in val_loader:
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item()
            
    print(f"Epoch {epoch+1} -> Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")

# --- FINAL TEST PHASE ---
# Run this ONLY after training is completely finished
model.eval()
# (Iterate through test_loader here to get final accuracy metrics)